# Welcome to Snowflake Notebooks in Workspaces! 🌟
Snowflake Notebooks are fully-managed Jupyter-powered notebook built for end-to-end DS and ML development on Snowflake data.

This includes:
- 🐍 **Familiar Jupyter experience** - Directly connected to the governed Snowflake data.
- 💻 **Inside Snowflake Workspaces** - Organize and run notebooks alongside your other files.
- 🧠 **Powerful for AI/ML** - Runs in our pre-built Container Runtime (CPU or GPU) with popular DS/ML packages pre-installed.
- 🏛️ **Governed collaboration** - Enable multiple users to collaborate via Git-backed Workspaces or Shared Workspaces.

## 1. Connect and run the notebook
Press [Run all] or [Connect] to create and connect your notebook to a service.

Optionally, run the below Python cell to inspect all pre-installed packages. To use a package, run `import <package_name>`. See example in "4. Visualize customer patterns". Learn more about [Managing packages and runtime](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-packages-runtime)

In [ ]:
!pip freeze # Shows all pre-installed packages

## 2. Create some customer data
This query generates a random table of customer data to explore

In [ ]:
%%sql -r dataframe_1
select
  seq4() as customer_id,
  'customer_' || (customer_id + 1) as name,
  'customer_' || (customer_id + 1) || '@example.com' as email,
  dateadd(day, -uniform(0, 3650, random()), current_date()) as signup_date,
  round(uniform(100, 100000, random()) / 100, 2) as lifetime_value
from table(generator(rowcount => 1000));

## 3. Find high value customers
This will show customers worth $800 or more. As you'll see, we're referencing the dataframe created above.

In [ ]:
%%sql -r dataframe_2
SELECT * FROM {{dataframe_1}} WHERE "LIFETIME_VALUE" > 800;

## 4. Visualize customer patterns
View the chart to see the distribution of customer tenture vs spend. Again, we're using the dataframe that was generated above.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from snowflake.snowpark import DataFrame as SnowparkDataFrame

# On runtime >=2.6 SQL cell results are returned as Snowpark DataFrames that need to be converted to pandas DataFrames
# On runtime <2.6 SQL cell results are directly returned as pandas DataFrames
# For more details, see https://docs.snowflake.com/en/developer-guide/snowflake-ml/container-runtime/releases
df = dataframe_1.to_pandas() if isinstance(dataframe_1, SnowparkDataFrame) else dataframe_1.copy()

df['SIGNUP_DATE'] = pd.to_datetime(df['SIGNUP_DATE'])
df['days_since_signup'] = (pd.Timestamp.today() - df['SIGNUP_DATE']).dt.days
df = df.sort_values('days_since_signup')

plt.figure(figsize=(10, 6))
plt.scatter(df['days_since_signup'], df['LIFETIME_VALUE'], alpha=0.3)
plt.plot(df['days_since_signup'], df['LIFETIME_VALUE'].rolling(100).mean(), color='red', linewidth=2, label='Rolling Avg')

plt.xlabel("Days Since Signup")
plt.ylabel("Lifetime Value")
plt.title("Customer Tenure vs Spend")
plt.legend()
plt.show()

# Congrats! Now create your own! ✅
Click "Create file" in a workspace and select Notebooks.

**Resources to help you get started**
- [YouTube Video Overview](https://www.youtube.com/watch?v=_kFhFIvnIrQ)
- [ML Quickstart using GPUs](https://www.snowflake.com/en/developers/guides/accelerate-topic-modeling-with-gpus-in-snowflake-ml/) - just download the .ipynb and upload to your Workspace!
- [Documentation](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-limitations)